In [1]:
import optuna
import axelrod
from axelrod.action import Action, actions_to_str
from axelrod.player import Player
from axelrod.strategy_transformers import (
    FinalTransformer,
    TrackHistoryTransformer,
)
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import numpy as np
from collections import Counter
import pandas as pd
from math import exp
import numpy as np


c:\Users\Ognjen\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
C, D = Action.C, Action.D

class FuzzyMethods():
    @staticmethod
    def calc_cooperation(self, opponent):
        return (Counter(opponent.history)[C])/len(opponent.history)*100
    
    @staticmethod
    def calc_adaptivity(self, opponent):
        adapCounter = 0
        adapReaction = 0

        if(len(self.history) < 3): 
            return 0
        
        for i in range(3, len(self.history)):
            if (self.history[i-3] == C and self.history[i-2] == D):
                adapCounter += 1
                if (opponent.history[i-1] == D):
                    adapReaction += 1
                elif (self.history[i-1] == D and opponent.history[i] == D):
                    adapReaction += 0.5
            elif (self.history[i-3] == D and self.history[i-2] == C):
                adapCounter += 1
                if (opponent.history[i-1] == C):
                    adapReaction += 1
                elif (self.history[i-1] == C and opponent.history[i] == C):
                    adapReaction += 0.5

        if adapCounter == 0:
            return 0
        
        return adapReaction/adapCounter*100

    
    @staticmethod
    def calc_forgiveness(self, opponent):
        DCounter = 0
        punishmentCounter = 0

        for i in range(0, len(self.history)-1):
            if(self.history[i] == D):
                DCounter += 1
                for j in range (i+1, len(opponent.history)):
                    if(opponent.history[j] == C):
                        break
                    else:
                        punishmentCounter += 1
              

        if punishmentCounter > 0:
            return DCounter/punishmentCounter*100
        else:
            return 100
    
    @staticmethod
    def calc_stochastic(self, opponent):
        patterns = [
            [C, C, C],
            [C, C, D],
            [C, D, C],
            [C, D, D],
            [D, C, C],
            [D, C, D],
            [D, D, C],
            [D, D, D]
        ]

        non_stochasticCounter = 0
        patternPlayedCounter = 0

        for p in patterns:
            opponentsReactions = []
            for i in range(0, len(self.history)-3):
                if ([self.history[i], self.history[i+1], self.history[i+2]] == p):
                    opponentsReactions.append([opponent.history[i+1], opponent.history[i+2], opponent.history[i+3]])
            
            unique_patterns = len(set(tuple(sub) for sub in opponentsReactions))

            if(len(opponentsReactions) > 0):
                non_stochasticCounter += 0 if unique_patterns == 1 else unique_patterns
                patternPlayedCounter += len(opponentsReactions)
        
        if patternPlayedCounter == 0:
            return 0
        
        return non_stochasticCounter/patternPlayedCounter*100
    

In [3]:
def build_fuzzy_player(params):
    _cooperation = ctrl.Antecedent(np.arange(0, 100, 1), 'cooperation')
    _adaptivity  = ctrl.Antecedent(np.arange(0, 100, 1), 'adaptivity')
    _forgiveness = ctrl.Antecedent(np.arange(0, 100, 1), 'forgiveness')
    _stochastic  = ctrl.Antecedent(np.arange(0, 100, 1), 'stochastic')
    _cooperation['low']    = fuzz.trimf(_cooperation.universe, [
        params['coop_low_a'], params['coop_low_b'], params['coop_low_c']
    ])
    _cooperation['medium'] = fuzz.trimf(_cooperation.universe, [
        params['coop_med_a'], params['coop_med_b'], params['coop_med_c']
    ])
    _cooperation['high']   = fuzz.trimf(_cooperation.universe, [
        params['coop_high_a'], params['coop_high_b'], params['coop_high_c']
    ])

    # --- _adaptivity (no, yes) ---
    _adaptivity['no']  = fuzz.trimf(_adaptivity.universe, [
        params['adap_no_a'], params['adap_no_b'], params['adap_no_c']
    ])
    _adaptivity['yes'] = fuzz.trimf(_adaptivity.universe, [
        params['adap_yes_a'], params['adap_yes_b'], params['adap_yes_c']
    ])

    # --- _forgiveness (low, medium, high) ---
    _forgiveness['low']    = fuzz.gaussmf(_forgiveness.universe, 0, params['forg_sigma'])
    _forgiveness['medium'] = fuzz.trimf(_forgiveness.universe, [
        params['forg_med_a'], params['forg_med_b'], params['forg_med_c']
    ])
    _forgiveness['high']   = fuzz.trimf(_forgiveness.universe, [
        params['forg_high_a'], params['forg_high_b'], params['forg_high_c']
    ])

    # --- _stochastic (none, sometimes, always) ---
    _stochastic['none']      = fuzz.trimf(_stochastic.universe, [
        params['stoch_none_a'], params['stoch_none_b'], params['stoch_none_c']
    ])
    _stochastic['sometimes'] = fuzz.trimf(_stochastic.universe, [
        params['stoch_some_a'], params['stoch_some_b'], params['stoch_some_c']
    ])
    _stochastic['always']    = fuzz.trimf(_stochastic.universe, [
        params['stoch_alw_a'], params['stoch_alw_b'], params['stoch_alw_c']
    ])

    # Resulting strategy MFs — also being optimized
    _resulting_strategy = ctrl.Consequent(np.arange(0, 100, 1), 'resulting_strategy')
    _resulting_strategy['D'] = fuzz.trimf(_resulting_strategy.universe, [
        params['D_a'],
        params['D_b'],
        params['D_c']
    ])
    _resulting_strategy['C'] = fuzz.trimf(_resulting_strategy.universe, [
        params['C_a'],
        params['C_b'],
        params['C_c']
    ])

    # Rebuild rules using the fresh variables above
    rule1 = ctrl.Rule(
        _cooperation['high'] & _adaptivity['no'] & (_forgiveness['medium'] | _forgiveness['high']),
        _resulting_strategy['D']
    )
    rule2 = ctrl.Rule(
        _forgiveness['low'] & _cooperation['high'],
        _resulting_strategy['C']
    )
    rule3 = ctrl.Rule(
        _stochastic['always'] | _adaptivity['no'],
        _resulting_strategy['D']
    )
    rule4 = ctrl.Rule(
        _cooperation['low'] | (_cooperation['medium'] & _forgiveness['low']),
        _resulting_strategy['D']
    )
    rule5 = ctrl.Rule(
        _cooperation['medium'] & _forgiveness['medium'] & _adaptivity['yes'],
        _resulting_strategy['C']
    )

    strategy_ctrl  = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5])
    _chosen_strategy = ctrl.ControlSystemSimulation(strategy_ctrl)

    # Build the player class dynamically, capturing everything in closure
    class OptimizedFuzzy(Player):

        # Override class-level FIS components with the fresh ones
        cooperation = _cooperation
        adaptivity = _adaptivity
        stochastic = _stochastic
        forgiveness = _forgiveness
        resulting_strategy = _resulting_strategy
        chosen_strategy = _chosen_strategy

        d_thresh = params['d_threshold']
        c_thresh = params['c_threshold']

        # Reset state so trials don't bleed into each other
        first_time = True
        h = {'Name': '', 'Fuzzy': [], 'Opponent': []}

        def strategy(self, opponent: axelrod.Player) -> Action:

            if len(self.history) == 0 or D not in opponent.history:
                return C

            coop  = FuzzyMethods.calc_cooperation(self, opponent)
            adap  = FuzzyMethods.calc_adaptivity(self, opponent)
            forg  = FuzzyMethods.calc_forgiveness(self, opponent)
            stoch = FuzzyMethods.calc_stochastic(self, opponent)

            self.chosen_strategy.input['cooperation'] = coop
            self.chosen_strategy.input['adaptivity']  = adap
            self.chosen_strategy.input['forgiveness'] = forg
            self.chosen_strategy.input['stochastic']  = stoch

            try:
                self.chosen_strategy.compute()
                output_val = self.chosen_strategy.output['resulting_strategy']
            except KeyError:
                # No rules fired — default to cooperate
                return C
            except Exception:
                return C

            d_membership = fuzz.interp_membership(
                self.resulting_strategy.universe,
                self.resulting_strategy['D'].mf,
                output_val
            )
            c_membership = fuzz.interp_membership(
                self.resulting_strategy.universe,
                self.resulting_strategy['C'].mf,
                output_val
            )

            if d_membership >= self.d_thresh and c_membership < self.c_thresh:
                return D

            return C

    return OptimizedFuzzy()

In [4]:
def sample_trimf(trial, name, universe_min, universe_max):
    """Sample a, b, c such that a <= b <= c is always guaranteed."""
    a = trial.suggest_int(f'{name}_a', universe_min, universe_max - 2)
    b = trial.suggest_int(f'{name}_b', a, universe_max - 1)
    c = trial.suggest_int(f'{name}_c', b, universe_max)
    return a, b, c


def objective(trial):

    # --- COOPERATION ---
    coop_low_a,  coop_low_b,  coop_low_c  = sample_trimf(trial, 'coop_low',   0, 50)
    coop_med_a,  coop_med_b,  coop_med_c  = sample_trimf(trial, 'coop_med',  15, 80)
    coop_high_a, coop_high_b, coop_high_c = sample_trimf(trial, 'coop_high', 50, 99)

    # --- ADAPTIVITY ---
    adap_no_a,  adap_no_b,  adap_no_c  = sample_trimf(trial, 'adap_no',   0, 60)
    adap_yes_a, adap_yes_b, adap_yes_c = sample_trimf(trial, 'adap_yes', 40, 99)

    # --- FORGIVENESS ---
    forg_sigma                          = trial.suggest_float('forg_sigma', 5, 40)
    forg_med_a,  forg_med_b,  forg_med_c  = sample_trimf(trial, 'forg_med',  10, 90)
    forg_high_a, forg_high_b, forg_high_c = sample_trimf(trial, 'forg_high', 60, 99)

    # --- STOCHASTIC ---
    stoch_none_a, stoch_none_b, stoch_none_c = sample_trimf(trial, 'stoch_none',  0, 50)
    stoch_some_a, stoch_some_b, stoch_some_c = sample_trimf(trial, 'stoch_some', 20, 80)
    stoch_alw_a,  stoch_alw_b,  stoch_alw_c  = sample_trimf(trial, 'stoch_alw',  55, 99)

    # --- OUTPUT ---
    D_a, D_b, D_c = sample_trimf(trial, 'D',  0, 60)
    C_a, C_b, C_c = sample_trimf(trial, 'C', 25, 99)

    # --- THRESHOLDS ---
    d_threshold = trial.suggest_float('d_threshold', 0.2, 0.7)
    c_threshold = trial.suggest_float('c_threshold', 0.3, 0.8)

    params = {
        'coop_low_a': coop_low_a, 'coop_low_b': coop_low_b, 'coop_low_c': coop_low_c,
        'coop_med_a': coop_med_a, 'coop_med_b': coop_med_b, 'coop_med_c': coop_med_c,
        'coop_high_a': coop_high_a, 'coop_high_b': coop_high_b, 'coop_high_c': coop_high_c,
        'adap_no_a': adap_no_a, 'adap_no_b': adap_no_b, 'adap_no_c': adap_no_c,
        'adap_yes_a': adap_yes_a, 'adap_yes_b': adap_yes_b, 'adap_yes_c': adap_yes_c,
        'forg_sigma': forg_sigma,
        'forg_med_a': forg_med_a, 'forg_med_b': forg_med_b, 'forg_med_c': forg_med_c,
        'forg_high_a': forg_high_a, 'forg_high_b': forg_high_b, 'forg_high_c': forg_high_c,
        'stoch_none_a': stoch_none_a, 'stoch_none_b': stoch_none_b, 'stoch_none_c': stoch_none_c,
        'stoch_some_a': stoch_some_a, 'stoch_some_b': stoch_some_b, 'stoch_some_c': stoch_some_c,
        'stoch_alw_a': stoch_alw_a, 'stoch_alw_b': stoch_alw_b, 'stoch_alw_c': stoch_alw_c,
        'D_a': D_a, 'D_b': D_b, 'D_c': D_c,
        'C_a': C_a, 'C_b': C_b, 'C_c': C_c,
        'd_threshold': d_threshold,
        'c_threshold': c_threshold,
    }

    try:
        fuzzy_player = build_fuzzy_player(params)
    except Exception as e:
        print(f"Failed to build player: {e}")
        return 0.0

    opponents = [player() for player in axelrod.stewart_plotkin_strategies]

    try:
        tournament = axelrod.Tournament(
            [fuzzy_player] + opponents,
            turns=200,
        )
        results = tournament.play(progress_bar=False)
    except Exception as e:
        print(f"Tournament failed: {e}")
        return 0.0

    return np.mean(results.normalised_scores[0])

In [5]:
study = optuna.create_study(
    direction='maximize',
    study_name='fuzzy_optimization_full',
    storage='sqlite:///fuzzy_optuna_full.db',
    load_if_exists=True
)

study.optimize(objective, n_trials=300, show_progress_bar=True)

print("\n=== OPTIMIZATION COMPLETE ===")
print(f"Best score:  {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

importance = optuna.importance.get_param_importances(study)
print("\n=== PARAMETER IMPORTANCE ===")
for param, imp in importance.items():
    print(f"  {param}: {imp:.4f}")

[I 2026-03-03 12:41:33,268] A new study created in RDB with name: fuzzy_optimization_full
Best trial: 0. Best value: 2.61818:   0%|          | 1/300 [00:32<2:43:17, 32.77s/it]

[I 2026-03-03 12:42:06,012] Trial 0 finished with value: 2.6181785714285715 and parameters: {'coop_low_a': 7, 'coop_low_b': 44, 'coop_low_c': 44, 'coop_med_a': 21, 'coop_med_b': 35, 'coop_med_c': 52, 'coop_high_a': 72, 'coop_high_b': 98, 'coop_high_c': 99, 'adap_no_a': 9, 'adap_no_b': 15, 'adap_no_c': 42, 'adap_yes_a': 81, 'adap_yes_b': 81, 'adap_yes_c': 88, 'forg_sigma': 33.93773553086606, 'forg_med_a': 73, 'forg_med_b': 78, 'forg_med_c': 78, 'forg_high_a': 65, 'forg_high_b': 74, 'forg_high_c': 99, 'stoch_none_a': 13, 'stoch_none_b': 37, 'stoch_none_c': 49, 'stoch_some_a': 25, 'stoch_some_b': 34, 'stoch_some_c': 65, 'stoch_alw_a': 86, 'stoch_alw_b': 88, 'stoch_alw_c': 94, 'D_a': 52, 'D_b': 59, 'D_c': 60, 'C_a': 30, 'C_b': 87, 'C_c': 97, 'd_threshold': 0.4393582177709032, 'c_threshold': 0.6140373220431714}. Best is trial 0 with value: 2.6181785714285715.


Best trial: 1. Best value: 2.63446:   1%|          | 2/300 [01:07<2:48:10, 33.86s/it]

[I 2026-03-03 12:42:40,661] Trial 1 finished with value: 2.6344642857142855 and parameters: {'coop_low_a': 45, 'coop_low_b': 45, 'coop_low_c': 45, 'coop_med_a': 77, 'coop_med_b': 77, 'coop_med_c': 77, 'coop_high_a': 87, 'coop_high_b': 87, 'coop_high_c': 91, 'adap_no_a': 32, 'adap_no_b': 48, 'adap_no_c': 51, 'adap_yes_a': 41, 'adap_yes_b': 64, 'adap_yes_c': 82, 'forg_sigma': 28.62695993034349, 'forg_med_a': 84, 'forg_med_b': 87, 'forg_med_c': 87, 'forg_high_a': 81, 'forg_high_b': 83, 'forg_high_c': 95, 'stoch_none_a': 34, 'stoch_none_b': 48, 'stoch_none_c': 49, 'stoch_some_a': 40, 'stoch_some_b': 58, 'stoch_some_c': 69, 'stoch_alw_a': 88, 'stoch_alw_b': 96, 'stoch_alw_c': 98, 'D_a': 1, 'D_b': 27, 'D_c': 27, 'C_a': 80, 'C_b': 94, 'C_c': 99, 'd_threshold': 0.32251206684152217, 'c_threshold': 0.4856206077578322}. Best is trial 1 with value: 2.6344642857142855.


Best trial: 1. Best value: 2.63446:   1%|          | 3/300 [01:31<2:25:58, 29.49s/it]

...

[I 2026-03-03 15:01:53,577] Trial 297 finished with value: 2.665964285714286 and parameters: {'coop_low_a': 21, 'coop_low_b': 38, 'coop_low_c': 43, 'coop_med_a': 32, 'coop_med_b': 33, 'coop_med_c': 66, 'coop_high_a': 66, 'coop_high_b': 83, 'coop_high_c': 89, 'adap_no_a': 39, 'adap_no_b': 40, 'adap_no_c': 41, 'adap_yes_a': 83, 'adap_yes_b': 88, 'adap_yes_c': 91, 'forg_sigma': 19.703566937423442, 'forg_med_a': 74, 'forg_med_b': 76, 'forg_med_c': 77, 'forg_high_a': 86, 'forg_high_b': 92, 'forg_high_c': 95, 'stoch_none_a': 37, 'stoch_none_b': 44, 'stoch_none_c': 46, 'stoch_some_a': 43, 'stoch_some_b': 63, 'stoch_some_c': 71, 'stoch_alw_a': 77, 'stoch_alw_b': 83, 'stoch_alw_c': 88, 'D_a': 5, 'D_b': 15, 'D_c': 16, 'C_a': 27, 'C_b': 65, 'C_c': 87, 'd_threshold': 0.4182733869045062, 'c_threshold': 0.3303439385526704}. Best is trial 282 with value: 2.7075714285714283.


Best trial: 282. Best value: 2.70757: 100%|█████████▉| 299/300 [2:20:48<00:27, 27.78s/it]

[I 2026-03-03 15:02:21,402] Trial 298 finished with value: 2.6806785714285715 and parameters: {'coop_low_a': 23, 'coop_low_b': 38, 'coop_low_c': 43, 'coop_med_a': 18, 'coop_med_b': 30, 'coop_med_c': 65, 'coop_high_a': 65, 'coop_high_b': 81, 'coop_high_c': 95, 'adap_no_a': 37, 'adap_no_b': 39, 'adap_no_c': 40, 'adap_yes_a': 65, 'adap_yes_b': 66, 'adap_yes_c': 95, 'forg_sigma': 21.05604253056892, 'forg_med_a': 14, 'forg_med_b': 15, 'forg_med_c': 44, 'forg_high_a': 87, 'forg_high_b': 92, 'forg_high_c': 95, 'stoch_none_a': 25, 'stoch_none_b': 44, 'stoch_none_c': 46, 'stoch_some_a': 75, 'stoch_some_b': 76, 'stoch_some_c': 77, 'stoch_alw_a': 76, 'stoch_alw_b': 91, 'stoch_alw_c': 94, 'D_a': 11, 'D_b': 16, 'D_c': 34, 'C_a': 34, 'C_b': 65, 'C_c': 77, 'd_threshold': 0.410335028777583, 'c_threshold': 0.34694932514643606}. Best is trial 282 with value: 2.7075714285714283.


Best trial: 282. Best value: 2.70757: 100%|██████████| 300/300 [2:21:17<00:00, 28.26s/it]


[I 2026-03-03 15:02:50,687] Trial 299 finished with value: 2.4959285714285713 and parameters: {'coop_low_a': 24, 'coop_low_b': 38, 'coop_low_c': 43, 'coop_med_a': 16, 'coop_med_b': 28, 'coop_med_c': 67, 'coop_high_a': 67, 'coop_high_b': 83, 'coop_high_c': 90, 'adap_no_a': 25, 'adap_no_b': 38, 'adap_no_c': 40, 'adap_yes_a': 63, 'adap_yes_b': 65, 'adap_yes_c': 86, 'forg_sigma': 19.178097776301797, 'forg_med_a': 55, 'forg_med_b': 72, 'forg_med_c': 80, 'forg_high_a': 87, 'forg_high_b': 92, 'forg_high_c': 96, 'stoch_none_a': 36, 'stoch_none_b': 43, 'stoch_none_c': 46, 'stoch_some_a': 49, 'stoch_some_b': 58, 'stoch_some_c': 72, 'stoch_alw_a': 77, 'stoch_alw_b': 90, 'stoch_alw_c': 94, 'D_a': 10, 'D_b': 14, 'D_c': 36, 'C_a': 45, 'C_b': 70, 'C_c': 87, 'd_threshold': 0.32142726693343326, 'c_threshold': 0.3175567598626865}. Best is trial 282 with value: 2.7075714285714283.

=== OPTIMIZATION COMPLETE ===
Best score:  2.7076
Best params: {'coop_low_a': 20, 'coop_low_b': 37, 'coop_low_c': 43, 'coop_